# Notebook 01 — Data Loading

**Day 1 deliverable.** Loads all raw datasets, verifies expected shapes and columns, checks for issues, and prints a data dictionary.
Run this notebook before any other.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
RAW  = ROOT / 'data' / 'raw'

def load_csv(path, **kwargs):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'Dataset not found: {p}\nDownload it first — see README.md')
    df = pd.read_csv(p, **kwargs)
    print(f'Loaded  {p.name:<40s}  {df.shape[0]} rows x {df.shape[1]} cols')
    return df

## Dataset 1 — kaushil268: Disease Prediction

Binary symptom matrix (132 symptom columns + `prognosis`). This is the primary dataset for Layer 1 modeling.

In [ ]:
train_df = load_csv(RAW / 'kaushil268' / 'Training.csv')
test_df  = load_csv(RAW / 'kaushil268' / 'Testing.csv')

print('\nTraining columns (first 10):', list(train_df.columns[:10]))
print('Unnamed columns:', [c for c in train_df.columns if 'Unnamed' in c])
print('Classes:', train_df['prognosis'].nunique()) 
train_df.head(3) 

Loaded  Training.csv                              4920 rows x 134 cols
Loaded  Testing.csv                               42 rows x 133 cols

Training columns (first 10): ['itching', 'skin_rash', 'nodal_skin_eruptions', 'continuous_sneezing', 'shivering', 'chills', 'joint_pain', 'stomach_pain', 'acidity', 'ulcers_on_tongue']
Unnamed columns: ['Unnamed: 133']
Classes: 41


,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis,Unnamed: 133
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN


In [3]:
print('Columns with embedded spaces:', [c for c in train_df.columns if ' ' in c])
print('NaN in Training:', train_df.isna().sum().sum(), '(should be only in Unnamed col)')
print('NaN in Testing: ', test_df.isna().sum().sum())
print('\nClass distribution (training):')
print(train_df['prognosis'].value_counts().to_string())

Columns with embedded spaces: ['spotting_ urination', 'foul_smell_of urine', 'dischromic _patches', 'Unnamed: 133']
NaN in Training: 4920 (should be only in Unnamed col)
NaN in Testing:  0

Class distribution (training):
prognosis
Fungal infection                           120
Allergy                                    120
GERD                                       120
Chronic cholestasis                        120
Drug Reaction                              120
Peptic ulcer diseae                        120
AIDS                                       120
Diabetes                                   120
Gastroenteritis                            120
Bronchial Asthma                           120
Hypertension                               120
Migraine                                   120
Cervical spondylosis                       120
Paralysis (brain hemorrhage)               120
Jaundice                                   120
Malaria                                    120
Chicken pox      

## Dataset 2 — itachi9604: Symptom–Precaution

Three files: symptom severity weights, disease descriptions, and precautions (the backbone of Layer 2).

In [4]:
itachi = RAW / 'itachi9604'

sev_df  = load_csv(itachi / 'Symptom-severity.csv')
desc_df = load_csv(itachi / 'symptom_Description.csv')
prec_df = load_csv(itachi / 'symptom_precaution.csv')

print('\nPrecaution columns:', list(prec_df.columns))
prec_df.head(5)

Loaded  Symptom-severity.csv                      133 rows x 2 cols
Loaded  symptom_Description.csv                   41 rows x 2 cols
Loaded  symptom_precaution.csv                    41 rows x 5 cols

Precaution columns: ['Disease', 'Precaution_1', 'Precaution_2', 'Precaution_3', 'Precaution_4']


,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Drug Reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,Malaria,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,Allergy,apply calamine,cover area with bandage,NaN,use ice to compress itching
3,Hypothyroidism,reduce stress,exercise,eat healthy,get proper sleep
4,Psoriasis,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths


In [5]:
print('Severity sample:')
print(sev_df.head(5).to_string(index=False))
print('\nDescription sample:')
print(desc_df.head(3).to_string(index=False))

Severity sample:
             Symptom  weight
             itching       1
           skin_rash       3
nodal_skin_eruptions       4
 continuous_sneezing       4
           shivering       5

Description sample:
      Disease                                                                                                                                                                                                                                    Description
Drug Reaction                                     An adverse drug reaction (ADR) is an injury caused by taking medication. ADRs may occur following a single dose or prolonged administration of a drug or result from the combination of two or more drugs.
      Malaria           An infectious disease caused by protozoan parasites from the Plasmodium family that can be transmitted by the bite of the Anopheles mosquito or by a contaminated needle or transfusion. Falciparum malaria is the most deadly type.
      Allergy An allergy is a

## Disease Overlap Check

Confirms that every disease in the training data has a corresponding row in the precaution table.

In [6]:
train_diseases = set(train_df['prognosis'].str.strip().str.lower())
prec_diseases  = set(prec_df['Disease'].str.strip().str.lower()) 

overlap = train_diseases & prec_diseases
print(f'kaushil268 diseases : {len(train_diseases)}')
print(f'itachi9604 diseases : {len(prec_diseases)}')
print(f'Overlap             : {len(overlap)}')

missing = sorted(train_diseases - prec_diseases)
if missing:
    print('Diseases with no precautions:', missing)
else:
    print('All training diseases have precautions. Treatment layer is fully covered.')

kaushil268 diseases : 41
itachi9604 diseases : 41
Overlap             : 41
All training diseases have precautions. Treatment layer is fully covered.


## Data Dictionary

| File | Key columns | Shape | Notes |
|---|---|---|---|
| Training.csv | 132 symptom cols + `prognosis` + `Unnamed:133` | 4920 rows | Drop Unnamed; binary (0/1) |
| Testing.csv | same minus Unnamed | 42 rows | 1 row per disease |
| Symptom-severity.csv | `Symptom`, `weight` | 133 rows | 1–7 scale |
| symptom_Description.csv | `Disease`, `Description` | 41 rows | One-line descriptions |
| symptom_precaution.csv | `Disease`, `Precaution_1`…`Precaution_4` | 41 rows | **Core of Layer 2** |

Proceed to `02_eda.ipynb`.